In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [2]:
BASE_DIR = Path.cwd().parent   # notebook içinden çalışıyorsan
DATA_PATH = BASE_DIR / "notebooks" / "df_combined.parquet"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_OUTPUT = MODEL_DIR / "behavioral_risk_regressor.joblib"

print("BASE_DIR:", BASE_DIR)
print("DATA_PATH:", DATA_PATH)
print("MODEL_OUTPUT:", MODEL_OUTPUT)

BASE_DIR: /Users/emre/code/emrekayaa/bank-shield-ai
DATA_PATH: /Users/emre/code/emrekayaa/bank-shield-ai/notebooks/df_combined.parquet
MODEL_OUTPUT: /Users/emre/code/emrekayaa/bank-shield-ai/models/behavioral_risk_regressor.joblib


In [3]:
#Yardımcı Fonksiyonlar Hücresi
def clean_to_float(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    ).fillna(0)


def build_customer_profiles(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    if "hour" not in df.columns and "date" in df.columns:
        df["hour"] = df["date"].dt.hour

    for col in ["amount", "yearly_income", "total_debt"]:
        if col in df.columns:
            df[col] = clean_to_float(df[col])
        else:
            df[col] = 0

    if "credit_score" not in df.columns:
        df["credit_score"] = 0

    if "is_fraud" not in df.columns:
        df["is_fraud"] = 0

    df["is_night_transaction"] = df["hour"].apply(
        lambda x: 1 if pd.notnull(x) and 0 <= x <= 6 else 0
    ) if "hour" in df.columns else 0

    if "date" in df.columns:
        df = df.sort_values(["client_id", "date"])
        df["fast_tx"] = (
            df.groupby("client_id")["date"]
            .diff()
            .dt.total_seconds()
            .lt(10)
            .fillna(False)
            .astype(int)
        )
    else:
        df["fast_tx"] = 0

    customer_df = df.groupby("client_id").agg({
        "amount": ["mean", "std", "max", "sum"],
        "is_fraud": "mean",
        "is_night_transaction": "mean",
        "fast_tx": "mean",
        "yearly_income": "first",
        "total_debt": "first",
        "credit_score": "first",
    })

    customer_df.columns = ["_".join(col).strip() for col in customer_df.columns.values]
    customer_df = customer_df.reset_index()

    customer_df["amount_std"] = customer_df["amount_std"].fillna(0)
    customer_df["yearly_income_first"] = customer_df["yearly_income_first"].replace(0, 1)

    customer_df["debt_to_income"] = (
        customer_df["total_debt_first"] / customer_df["yearly_income_first"]
    )

    return customer_df


def minmax_0_100(series: pd.Series) -> pd.Series:
    s = series.copy()
    s_min = s.min()
    s_max = s.max()
    if s_max == s_min:
        return pd.Series(np.full(len(s), 50.0), index=s.index)
    return ((s - s_min) / (s_max - s_min)) * 100


def build_continuous_target(customer_df: pd.DataFrame) -> pd.DataFrame:
    df = customer_df.copy()

    dti_component = np.log1p(df["debt_to_income"].clip(lower=0))
    dti_score = minmax_0_100(dti_component)

    credit_clipped = df["credit_score_first"].clip(lower=300, upper=850)
    credit_inverse = 850 - credit_clipped
    credit_score_component = minmax_0_100(credit_inverse)

    fraud_component = minmax_0_100(df["is_fraud_mean"].clip(lower=0))
    night_component = minmax_0_100(df["is_night_transaction_mean"].clip(lower=0))
    fast_component = minmax_0_100(df["fast_tx_mean"].clip(lower=0))
    amount_component = minmax_0_100(np.log1p(df["amount_mean"].clip(lower=0)))

    df["risk_score_target"] = (
        0.45 * dti_score +
        0.25 * credit_score_component +
        0.12 * fraud_component +
        0.08 * night_component +
        0.05 * fast_component +
        0.05 * amount_component
    )

    df["risk_score_target"] = df["risk_score_target"].clip(0, 100)

    return df


def get_risk_group(score: float) -> str:
    if score < 30:
        return "DÜŞÜK"
    elif score < 70:
        return "ORTA"
    return "YÜKSEK"

In [4]:
#Veriyi okuma
print("Veri okunuyor...")
df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()

Veri okunuyor...
(13305915, 46)


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,year_pin_last_changed,card_on_dark_web,mcc_description,is_fraud,year,month,day,hour,day_of_week,is_weekend
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,...,2008,No,Miscellaneous Food Stores,0,2010,1,1,0,Friday,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,2015,No,Department Stores,0,2010,1,1,0,Friday,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,2008,No,Money Transfer,0,2010,1,1,0,Friday,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,2006,No,Money Transfer,0,2010,1,1,0,Friday,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,2014,No,Drinking Places (Alcoholic Beverages),0,2010,1,1,0,Friday,0


In [5]:
#Müşteri Profili Oluşturma
print("Müşteri profilleri oluşturuluyor...")
customer_df = build_customer_profiles(df)

print(customer_df.shape)
customer_df.head()

Müşteri profilleri oluşturuluyor...
(1219, 12)


,client_id,amount_mean,amount_std,amount_max,amount_sum,is_fraud_mean,is_night_transaction_mean,fast_tx_mean,yearly_income_first,total_debt_first,credit_score_first,debt_to_income
0,0,61.033190,76.355839,1128.47,780919.67,0.000469,0.043454,0.004846,59613,36199,763,0.607233
1,1,36.525501,49.982647,937.15,367921.37,0.001291,0.004864,0.002482,45360,14587,704,0.321583
2,100,77.307830,103.795106,1639.74,546566.36,0.005092,0.104809,0.010184,48944,79960,813,1.633704
3,1002,75.155505,92.928943,876.55,279879.10,0.002148,0.106337,0.008593,38788,77448,716,1.996700
4,1003,99.118991,112.977817,3077.02,1073458.67,0.000462,0.020868,0.002678,80526,117380,632,1.457666


In [6]:
#Sürekli Target Oluşturma
print("Sürekli hedef skor oluşturuluyor...")
customer_df = build_continuous_target(customer_df)

customer_df[[
    "client_id",
    "debt_to_income",
    "credit_score_first",
    "risk_score_target"
]].head()

Sürekli hedef skor oluşturuluyor...


,client_id,debt_to_income,credit_score_first,risk_score_target
0,0,0.607233,763,23.341267
1,1,0.321583,704,21.261844
2,100,1.633704,813,37.851257
3,1002,1.996700,716,45.604045
4,1003,1.457666,632,43.480284


In [7]:
#Feature ve Target ayırma
feature_cols = [
    "amount_mean",
    "amount_std",
    "amount_max",
    "amount_sum",
    "is_night_transaction_mean",
    "fast_tx_mean",
    "yearly_income_first",
    "total_debt_first",
    "credit_score_first",
]

X = customer_df[feature_cols].copy()
y = customer_df["risk_score_target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1219, 9)
y shape: (1219,)


In [8]:
#Train - Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (975, 9)
X_test : (244, 9)


In [9]:
#Model eğitme
print("Model eğitiliyor...")

model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

model.fit(X_train, y_train)

Model eğitiliyor...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [10]:
#Tahmin ve Metrikler
y_pred = model.predict(X_test).clip(0, 100)

print("Model performansı")
print("-" * 30)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R2 : {r2_score(y_test, y_pred):.3f}")

Model performansı
------------------------------
MAE: 1.495
R2 : 0.963


In [11]:
#Örnek Sonuçlar
preview = pd.DataFrame({
    "Gerçek Skor": y_test.values[:10],
    "Tahmin Skor": y_pred[:10],
    "Tahmin Grup": [get_risk_group(v) for v in y_pred[:10]]
})

preview

,Gerçek Skor,Tahmin Skor,Tahmin Grup
0,46.765083,46.870865,ORTA
1,58.230398,55.332661,ORTA
2,42.850778,44.532600,ORTA
3,34.935346,37.133682,ORTA
4,39.518481,36.999603,ORTA
5,25.471802,25.495651,DÜŞÜK
6,32.331150,34.118313,ORTA
7,45.579663,47.141411,ORTA
8,31.197975,31.536133,ORTA
9,42.506754,42.381565,ORTA


In [12]:
#Modeli kaydetme
payload = {
    "model": model,
    "features": feature_cols,
    "model_type": "regressor",
    "target_name": "risk_score_target",
    "notes": "Continuous 0-100 behavioral risk score model"
}

joblib.dump(payload, MODEL_OUTPUT)
print(f"Model kaydedildi: {MODEL_OUTPUT}")


Model kaydedildi: /Users/emre/code/emrekayaa/bank-shield-ai/models/behavioral_risk_regressor.joblib
